# WP-T full-secret run on Colab (ML-DSA-65 / -87)

Runs `experiments/full_secret_real.py` on a Colab GPU for the two large parameter sets that are
too heavy for the laptop. Results save to Google Drive and survive a disconnect (the script
resumes). Same seed as the laptop run, so key `k` is the *same* secret and challenge set on every
machine; shards merge cleanly.

**Private.** This uploads the project code, including the unpublished Kao P2 attack. Keep this
notebook and its Drive folder private; do not share the link. (Project rule: ask before putting
the repo anywhere; this is that step.)

**Before you start:** Runtime -> Change runtime type -> **A100 GPU** (needs Colab Pro+ for the
high-RAM A100; T4 works but is slower and lower-RAM).

## One-time prep on the laptop
Make a code zip (no results, no PDFs) and drop it in Drive at `MyDrive/talus/project.zip`:
```bash
cd "<project root>"
zip -r project.zip harness experiments -x '*/__pycache__/*' -x 'experiments/results/*' -x 'experiments/colab/*'
```
Then upload `project.zip` to `MyDrive/talus/` in your browser.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA:', torch.cuda.is_available(), torch.__version__)

In [ ]:
# 2. Mount Drive and unpack the code into /content/project
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, pathlib
DRIVE = '/content/drive/MyDrive/talus'
PROJ  = '/content/project'
os.makedirs(PROJ, exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/project.zip') as z:
    z.extractall(PROJ)
print('unpacked:', os.listdir(PROJ))

In [ ]:
# 3. Deps. Colab already has numpy, scipy, torch (CUDA). Install nothing unless an import fails.
import numpy, scipy, torch
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| torch', torch.__version__)

In [ ]:
# 4. Pick the set and the key range for THIS machine, and run.
#    Results go to Drive so a disconnect only loses the row in progress (the script resumes).
#    Split example: laptop does keys 0-15, this Colab does 15-30 -> set KEY_START=15, KEY_END=30.
PARAMS    = 'ML-DSA-65'          # or 'ML-DSA-87'
N_GRID    = '130000 143000 160000 179000 215000'   # -65 grid; -87: 120000 135000 148000 165000 184000
KEYS      = 30
KEY_START = 0
KEY_END   = 30
SEED      = 3001                 # MUST match the laptop run so shards align
TIME_LIMIT= 180                  # ILP cap per element, seconds (A100 can afford more than the laptop's 120)

span = '' if (KEY_START == 0 and KEY_END == KEYS) else f'_k{KEY_START}-{KEY_END}'
OUT  = f'{DRIVE}/results/full_secret_real_{PARAMS}{span}.json'
import subprocess, sys
cmd = [sys.executable, '-u', '-m', 'experiments.full_secret_real',
       '--params', PARAMS, '--N', *N_GRID.split(), '--keys', str(KEYS),
       '--key-start', str(KEY_START), '--key-end', str(KEY_END),
       '--seed', str(SEED), '--device', 'gpu', '--time-limit', str(TIME_LIMIT),
       '--out', OUT]
print(' '.join(cmd))
# stream output live; rows are also written to OUT on Drive after each one
proc = subprocess.Popen(cmd, cwd=PROJ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait(); print('exit', proc.returncode)

## After the run
The shard JSON is in `MyDrive/talus/results/`. Download it to the laptop's
`experiments/results/` and merge all shards (laptop + Colab) into one summary:
```bash
python -m experiments.merge_full_secret --params ML-DSA-65
```
It checks the seed matches across shards before merging.